# Day 67: Regularization in Deep Learning - Dropout & BatchNorm

## The Overfitting Problem in Deep Nets
Deep networks have a huge number of parameters, which gives them the capacity to memorize the training set exactly instead of learning patterns that generalize to new data - this shows up as high training accuracy but poor performance on unseen data. Regularization is any technique that limits this memorization so the network is forced to learn more robust, general features. In this notebook we look at three of the most common regularization tools in deep learning: Dropout, Batch Normalization, and Early Stopping. The code cell below just sets up a small synthetic binary classification dataset (10 random features, label based on the first two) that we'll reuse to demonstrate each technique.</cell id="cell-1">


In [ ]:
# pip install tensorflow
import numpy as np
from tensorflow import keras
from tensorflow.keras import layers

np.random.seed(0)  # reproducible random data
X = np.random.randn(300, 10)  # 300 samples, 10 random features
y = (X[:, 0] + X[:, 1] > 0).astype(int)  # label depends only on the first two features; the rest are noise

## Dropout
Dropout is a regularization layer that, during training, randomly "switches off" a fraction of neurons on each forward pass (here 30%, via `Dropout(0.3)`), so those neurons' outputs are ignored and their weights don't get updated for that step. This prevents the network from becoming overly dependent on any single neuron or small group of neurons to make its prediction, which forces it to spread useful information across many neurons and learn more redundant, general-purpose features. Dropout is only active during training - at inference/prediction time, Keras automatically uses all neurons (scaled appropriately), so no extra code is needed to disable it. It's one of the simplest and most widely used regularizers in deep learning, especially for large fully-connected networks that are prone to overfitting.</cell id="cell-3">


In [ ]:
model = keras.Sequential([
    layers.Dense(64, activation='relu', input_shape=(10,)),
    layers.Dropout(0.3),  # randomly zeroes 30% of this layer's outputs during training only
    layers.Dense(32, activation='relu'),
    layers.Dropout(0.3),  # dropout again after the second hidden layer
    layers.Dense(1, activation='sigmoid')  # binary output
])
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

## Batch Normalization
Batch Normalization (BatchNorm) normalizes the outputs of a layer - for each mini-batch, it rescales the activations to have roughly zero mean and unit variance, then applies its own small learnable scale and shift. Without it, activations can drift to very large or very small values as they pass through many layers, which slows down training and makes the network sensitive to how weights are initialized and how large the learning rate is. By keeping activations in a stable, consistent range, BatchNorm lets you train faster (often with a higher learning rate) and acts as a mild regularizer too, since the batch statistics add a bit of noise similar in spirit to dropout. Note the layer order below: the Dense layer produces raw (linear) outputs first, BatchNormalization normalizes them, and only then does the Activation('relu') apply the nonlinearity - this Dense -> BatchNorm -> Activation ordering is the standard recipe.</cell id="cell-5">


In [ ]:
model_bn = keras.Sequential([
    layers.Dense(64, input_shape=(10,)),  # no activation here yet - BatchNorm runs on the raw linear output
    layers.BatchNormalization(),  # normalizes activations to stabilize and speed up training
    layers.Activation('relu'),  # nonlinearity applied after normalization
    layers.Dense(1, activation='sigmoid')
])
model_bn.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

## Early Stopping
Even with dropout and batch normalization, training for too many epochs can still overfit, since the model keeps improving on the training set long after it has stopped improving on unseen data. Early stopping is a callback that monitors a chosen metric (here `val_loss`, the loss on the held-out validation split) after every epoch, and halts training once it fails to improve for a set number of epochs in a row (`patience=3`). `restore_best_weights=True` is important - it means that when training stops, the model's weights are rolled back to whichever epoch had the best validation loss, not just wherever training happened to stop. This turns "how many epochs to train for" from a manual guess into something the training process figures out automatically.</cell id="cell-7">


In [ ]:
from tensorflow.keras.callbacks import EarlyStopping

# stop training if val_loss doesn't improve for 3 consecutive epochs, and keep the best weights seen so far
early_stop = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)
model.fit(X, y, epochs=50, validation_split=0.2, callbacks=[early_stop], verbose=0)  # may stop well before 50 epochs
print('Training stopped early if no improvement for 3 epochs')